In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import os
import midas.file_reader
from datetime import datetime


# utill

def lengths_above_threshold(x: np.ndarray, threshold: float) -> np.ndarray:

    if x.ndim != 1:
        raise ValueError("L'array deve essere 1D")

    above = x > threshold
    if not np.any(above):
        return np.array([], dtype=int), np.array([], dtype=int)

    d = np.diff(above.astype(np.int8))

    starts = np.where(d == 1)[0] + 1
    ends   = np.where(d == -1)[0] + 1

    # Gestione bordi
    if above[0]:
        starts = np.r_[0, starts]
    if above[-1]:
        ends = np.r_[ends, len(x)]

    lengths = ends - starts

    return starts, lengths

# script settings
verbose = False
DEBUG   = False
outplot = False 

# load file
run     = int(input("run number [int] "))
path    = '/home/cold/data/'    
fname = ('/run%05d.mid.gz' % run)
mf = midas.file_reader.MidasFile(path+fname)

# get odb dump and retrive info  #######
odb = mf.get_bor_odb_dump().data

try:
    Run_description   = odb['Experiment']['Run Parameters']['Run description']
    Events_logged     = odb['Logger']['Channels']['0']['Statistics']['Events written']
    
    print('Run_description: ', Run_description)
    print('Events_logged: ', Events_logged)
except:
    print('WARNING: no run description')

# Spectrum statis setup (to be retrive from ODB in the future)
ADCmax = 2**16
inputRange = 5
Nch = 8
fs = 5e6

# loop on event

waveform = []
nbuffer  = 0
tr_time_p = 0
nbuffer_p = 0
tr_sample_p = 0

for event in mf:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if verbose or event_number % 1000==0:
        print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("Event # %s at %s, banks %s" % (event_number, datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))

    for bank_name, bank in event.banks.items():
        if ('SPEC' in bank_name): 
#             # print(event.banks['SPEC'].data[:10])
#             data=np.array(event.banks['SPEC'].data)
#             Nsamp = data.size // Nch
#             data = data.reshape(Nsamp, Nch)
#             #data = data/(ADCmax/2)*inputRange
#             data = data.astype(np.float64) * (2 * inputRange / 65536.0)
#             # print(data[10000,0])
            


            u = np.asarray(event.banks['SPEC'].data, dtype=np.uint16)  # 0..65535
            s = u.view(np.int16)                                       # -32768..32767
            Nsamp = s.size // Nch
            frames = s[:Nsamp * Nch].reshape(Nsamp, Nch)  # [time, ch]
            data = frames.astype(np.float64) * (2.0 * inputRange / 65536.0)
            
            t = np.arange(Nsamp) / fs  # [s]
            # serch second in channel 6
            start, length=lengths_above_threshold(data[:, 6], 2.5) 
            print(start,length)
            print('Lunghezza Buffer n',nbuffer,':',len(data[:, 6]))
            if len(length)>0:
                tr_second = np.argmax(length > 4000)
                if tr_second>0:
                    buffer_len = len(data[:, 6])
                    tr_sample  = len(data[:, 6])*nbuffer+start[tr_second]
                    tr_time    = tr_sample/fs
                    print(event.header.timestamp, nbuffer, nbuffer - nbuffer_p, buffer_len,  
                        start[tr_second], length[tr_second], tr_sample,  tr_sample-tr_sample_p,
                        tr_time, tr_time-tr_time_p)
                    
                    tr_time_p   = tr_time
                    tr_sample_p = tr_sample
                    nbuffer_p   = nbuffer
                    
                    #
                    fig, ax = plt.subplots(2,1, figsize=(8, 4))
                    ax[0].plot(t[start[tr_second]-100:start[tr_second]+length[tr_second]+100], 
                             data[start[tr_second]-100:start[tr_second]+length[tr_second]+100, 6])
                    ax[1].plot(t[start[tr_second]-1000:start[tr_second]+1000], 
                             data[start[tr_second]-1000:start[tr_second]+1000, 0])
                    ax[0].set_ylabel("CH6")
                    ax[1].set_ylabel("CH0")
                    ax[0].grid(True)
                    ax[1].grid(True)
                    fig.tight_layout()
                    plt.show()      
                    #

                    
            nbuffer+=1


            
            if outplot:
                fig1, ax1 = plt.subplots(2, Nch//2, figsize=(8, 4), sharex=True)
                for ch in range(Nch):
                    row = ch // 4
                    col = ch % 4  

                    ax1[row, col].plot(t, data[:, ch])
                    ax1[row, col].set_ylabel(f"CH{ch}")
                    ax1[row, col].grid(True)

                fig1.tight_layout()
                plt.show()         

                fig2, ax2 = plt.subplots(figsize=(8, 2), sharex=True)
                ax2.plot(t[:10000], data[:10000, 6])
                ax2.set_ylabel("CH6")
                ax2.grid(True)
                fig2.tight_layout()
                plt.show()          # mostra fig2
            
    if DEBUG:        
        if event.header.serial_number == 20: # si ferm dopo i primi 20 eventi
            break
            
print('DONE')


In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

c = np.concatenate((a, b))
print(c)
print(c[:2])

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import os
import midas.file_reader
from datetime import datetime


# utill

def lengths_above_threshold(x: np.ndarray, threshold: float) -> np.ndarray:

    if x.ndim != 1:
        raise ValueError("L'array deve essere 1D")

    above = x > threshold
    if not np.any(above):
        return np.array([], dtype=int), np.array([], dtype=int)

    d = np.diff(above.astype(np.int8))

    starts = np.where(d == 1)[0] + 1
    ends   = np.where(d == -1)[0] + 1

    # Gestione bordi
    if above[0]:
        starts = np.r_[0, starts]
    if above[-1]:
        ends = np.r_[ends, len(x)]

    lengths = ends - starts

    return starts, lengths

# script settings
verbose = False
DEBUG   = False
outplot = False 

# load file
run     = int(input("run number [int] "))
path    = '/home/cold/data/'    
fname = ('/run%05d.mid.gz' % run)
mf = midas.file_reader.MidasFile(path+fname)

# get odb dump and retrive info  #######
odb = mf.get_bor_odb_dump().data

try:
    Run_description   = odb['Experiment']['Run Parameters']['Run description']
    Events_logged     = odb['Logger']['Channels']['0']['Statistics']['Events written']
    
    print('Run_description: ', Run_description)
    print('Events_logged: ', Events_logged)
except:
    print('WARNING: no run description')

# Spectrum statis setup (to be retrive from ODB in the future)
ADCmax = 2**16
inputRange = 5
Nch = 8
fs = 5e6

# loop on event

waveform    = []
nbuffer     = 0
tr_time_p   = 0
nbuffer_p   = 0
tr_sample_p = 0
d_size      = 8000
left_data   = np.zeros(d_size)

for event in mf:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if verbose or event_number % 1000==0:
        print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("Event # %s at %s, banks %s" % (event_number, datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))

    for bank_name, bank in event.banks.items():
        if ('SPEC' in bank_name): 
            u = np.asarray(event.banks['SPEC'].data, dtype=np.uint16)  # 0..65535
            s = u.view(np.int16)                                       # -32768..32767
            Nsamp = s.size // Nch
            frames = s[:Nsamp * Nch].reshape(Nsamp, Nch)  # [time, ch]
            data = frames.astype(np.float64) * (2.0 * inputRange / 65536.0)
            t = np.arange(Nsamp) / fs  # [s]
            
            
            right_data = data[:d_size, 0]
            j_data = np.concatenate((left_data, right_data))
            j_time = np.linspace(-len(j_data)/2,len(j_data)/2, len(j_data))
            #

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(j_time, j_data)
            fig.tight_layout()
            plt.show()
            #
            start, length=lengths_above_threshold(data[:, 6], 2.5) # 2.5 V
            if len(length)>0:
                tr_second = np.argmax(length > 4000) # lunghezzo 4000 sample
                if tr_second>0:
                    buffer_len = len(data[:, 6])
                    tr_sample  = len(data[:, 6])*nbuffer+start[tr_second]
                    tr_time    = tr_sample/fs
                    print(event.header.timestamp, nbuffer, nbuffer - nbuffer_p, buffer_len,  
                        start[tr_second], length[tr_second], tr_sample,  tr_sample-tr_sample_p,
                        tr_time, tr_time-tr_time_p)
                    
                    print(start, length)
                    
                    if (tr_sample-tr_sample_p)>5000001:
                        #
                        prezoom = 1000
                        postzoom = 100
                        fig, ax = plt.subplots(2,1, figsize=(8, 4))
                        ax[0].plot(t[start[tr_second]-prezoom:start[tr_second]+length[tr_second]+postzoom], 
                                 data[start[tr_second]-prezoom:start[tr_second]+length[tr_second]+postzoom, 6])
                        ax[1].plot(t[start[tr_second]-prezoom:start[tr_second]+postzoom], 
                                 data[start[tr_second]-prezoom:start[tr_second]+postzoom, 0])
                        ax[0].set_ylabel("CH6")
                        ax[1].set_ylabel("CH0")
                        ax[0].grid(True)
                        ax[1].grid(True)
                        fig.tight_layout()
                        plt.show()    
                        #
                    
                    
                    tr_time_p   = tr_time
                    tr_sample_p = tr_sample
                    # end
                    
            nbuffer_p = nbuffer    
            nbuffer  += 1
            left_data = data[-d_size:, 0]


            
            if outplot:
#                 fig1, ax1 = plt.subplots(2, Nch//2, figsize=(8, 4), sharex=True)
#                 for ch in range(Nch):
#                     row = ch // 4
#                     col = ch % 4  
                    
#                     ax1[row, col].plot(t, data[:, ch])
#                     ax1[row, col].set_ylabel(f"CH{ch}")
#                     ax1[row, col].grid(True)
                
#                 fig1.tight_layout()
#                 plt.show()         
                fig1, ax1 = plt.subplots(Nch, 1, figsize=(8, 2*Nch), sharex=True)

                # Se Nch == 1 matplotlib non restituisce un array
                if Nch == 1:
                    ax1 = [ax1]

                for ch in range(Nch):
                    ax1[ch].plot(t[:10000], data[:10000, ch])
                    ax1[ch].set_ylabel(f"CH{ch}")
                    ax1[ch].grid(True)

                ax1[-1].set_xlabel("Time")

                fig1.tight_layout()
                plt.show()
                
                fig2, ax2 = plt.subplots(figsize=(8, 2), sharex=True)
                ax2.plot(t[:10000], data[:10000, 6])
                ax2.set_ylabel("CH6")
                ax2.grid(True)
                fig2.tight_layout()
                plt.show()          # mostra fig2
            
    if DEBUG:        
        if event.header.serial_number == 20: # si ferm dopo i primi 20 eventi
            break
            
print('DONE')


In [ ]:
raw_data = """
1771948280 7 7 655360 412521 4991 5000041 5000000 1.0000082 0.9999999999999999
1771948281 15 8 655360 169641 4991 10000041 5000000 2.0000082 1.0
1771948282 22 7 655360 582121 4991 15000041 5000000 3.0000082 1.0
1771948283 30 8 655360 339242 4990 20000042 5000001 4.0000084 1.0000001999999997
1771948285 38 8 655360 96362 4991 25000042 5000000 5.0000084 1.0
1771948285 45 7 655360 508842 4991 30000042 5000000 6.0000084 1.0
1771948287 53 8 655360 265962 4991 35000042 5000000 7.0000084 1.0
1771948287 61 8 655360 23083 4990 40000043 5000001 8.0000086 1.0000001999999997
1771948289 68 7 655360 435563 4991 45000043 5000000 9.0000086 1.0
1771948289 76 8 655360 192683 4991 50000043 5000000 10.0000086 1.0
1771948290 83 7 655360 605163 4991 55000043 5000000 11.0000086 1.0
1771948291 91 8 655360 362283 4991 60000043 5000000 12.0000086 1.0
1771948292 99 8 655360 119404 4991 65000044 5000001 13.0000088 1.0000002000000006
1771948294 107 8 655360 269740 4991 70393260 5393216 14.078652 1.0786432000000001
1771948294 115 8 655360 26860 4991 75393260 5000000 15.078652 1.0
1771948296 122 7 655360 439340 4991 80393260 5000000 16.078652 1.0000000000000018
1771948296 130 8 655360 196461 4990 85393261 5000001 17.0786522 1.0000001999999988
1771948297 137 7 655360 608941 4991 90393261 5000000 18.0786522 1.0
1771948298 145 8 655360 366061 4991 95393261 5000000 19.0786522 1.0
1771948299 153 8 655360 123181 4991 100393261 5000000 20.0786522 1.0
1771948300 160 7 655360 535661 4991 105393261 5000000 21.0786522 1.0
1771948301 168 8 655360 292782 4990 110393262 5000001 22.0786524 1.0000001999999988
1771948303 176 8 655360 49902 4991 115393262 5000000 23.0786524 1.0
1771948303 183 7 655360 462382 4991 120393262 5000000 24.0786524 1.0
1771948305 191 8 655360 219502 4991 125393262 5000000 25.0786524 1.0
1771948305 198 7 655360 631983 4990 130393263 5000001 26.0786526 1.0000002000000023
1771948307 207 9 655360 126959 4991 135786479 5393216 27.1572958 1.0786431999999984
1771948307 214 7 655360 539439 4991 140786479 5000000 28.1572958 1.0
1771948309 222 8 655360 296559 4991 145786479 5000000 29.1572958 1.0
1771948309 230 8 655360 53679 4991 150786479 5000000 30.1572958 1.0
1771948311 237 7 655360 466160 4990 155786480 5000001 31.157296 1.0000001999999988
1771948311 245 8 655360 223280 4991 160786480 5000000 32.157296 1.0000000000000036
1771948312 252 7 655360 635760 4991 165786480 5000000 33.157296 1.0
1771948313 260 8 655360 392880 4991 170786480 5000000 34.157296 1.0
1771948314 268 8 655360 150001 4990 175786481 5000001 35.1572962 1.0000001999999952
1771948315 275 7 655360 562481 4991 180786481 5000000 36.1572962 1.0
1771948316 283 8 655360 319601 4991 185786481 5000000 37.1572962 1.0
1771948318 291 8 655360 76721 4991 190786481 5000000 38.1572962 1.0
1771948318 298 7 655360 489201 4991 195786481 5000000 39.1572962 1.0
1771948320 306 8 655360 246322 4991 200786482 5000001 40.1572964 1.0000002000000023
1771948320 314 8 655360 396658 4991 206179698 5393216 41.2359396 1.078643200000002
1771948322 322 8 655360 153778 4991 211179698 5000000 42.2359396 1.0
1771948322 329 7 655360 566258 4991 216179698 5000000 43.2359396 1.0
1771948324 337 8 655360 323379 4990 221179699 5000001 44.2359398 1.0000001999999952
1771948324 345 8 655360 80499 4991 226179699 5000000 45.2359398 1.0
1771948326 352 7 655360 492979 4991 231179699 5000000 46.2359398 1.0
1771948326 360 8 655360 250099 4991 236179699 5000000 47.2359398 1.0
1771948327 368 8 655360 7219 4991 241179699 5000000 48.2359398 1.0
1771948328 375 7 655360 419700 4991 246179700 5000001 49.23594 1.0000002000000023
1771948329 383 8 655360 176820 4991 251179700 5000000 50.23594 1.0
1771948330 390 7 655360 589300 4991 256179700 5000000 51.23594 1.0
1771948331 398 8 655360 346420 4991 261179700 5000000 52.23594 1.0
1771948333 406 8 655360 103541 4990 266179701 5000001 53.2359402 1.0000002000000023
1771948333 414 8 655360 253877 4991 271572917 5393216 54.3145834 1.0786431999999948
1771948335 422 8 655360 10997 4991 276572917 5000000 55.3145834 1.0
1771948335 429 7 655360 423477 4991 281572917 5000000 56.3145834 1.0
1771948337 437 8 655360 180597 4991 286572917 5000000 57.3145834 1.0
1771948337 444 7 655360 593078 4991 291572918 5000001 58.3145836 1.0000002000000023
1771948339 452 8 655360 350198 4991 296572918 5000000 59.3145836 1.0
1771948339 460 8 655360 107318 4991 301572918 5000000 60.3145836 1.0
1771948340 467 7 655360 519798 4991 306572918 5000000 61.3145836 1.0
1771948341 475 8 655360 276919 4990 311572919 5000001 62.3145838 1.0000002000000023
1771948342 483 8 655360 34039 4991 316572919 5000000 63.3145838 1.0
1771948343 490 7 655360 446519 4991 321572919 5000000 64.3145838 0.9999999999999929
1771948344 498 8 655360 203639 4991 326572919 5000000 65.3145838 1.0
1771948345 505 7 655360 616119 4991 331572919 5000000 66.3145838 1.0
1771948346 513 8 655360 373240 4991 336572920 5000001 67.314584 1.0000002000000023
1771948347 521 8 655360 523576 4991 341966136 5393216 68.3932272 1.078643200000002
1771948348 529 8 655360 280696 4991 346966136 5000000 69.3932272 1.0
1771948350 537 8 655360 37816 4991 351966136 5000000 70.3932272 1.0
1771948350 544 7 655360 450297 4990 356966137 5000001 71.3932274 1.0000002000000023
1771948352 552 8 655360 207417 4991 361966137 5000000 72.3932274 1.0
1771948352 559 7 655360 619897 4991 366966137 5000000 73.3932274 1.0
1771948354 567 8 655360 377017 4991 371966137 5000000 74.3932274 1.0
1771948354 575 8 655360 134137 4991 376966137 5000000 75.3932274 1.0
1771948355 582 7 655360 546618 4991 381966138 5000001 76.3932276 1.0000002000000023
1771948356 590 8 655360 303738 4991 386966138 5000000 77.3932276 1.0
1771948357 598 8 655360 60858 4991 391966138 5000000 78.3932276 1.0
1771948358 605 7 655360 473338 4991 396966138 5000000 79.3932276 1.0
1771948359 613 8 655360 230459 4990 401966139 5000001 80.3932278 1.0000002000000023
1771948360 621 8 655360 380795 4991 407359355 5393216 81.471871 1.0786431999999877
1771948361 629 8 655360 137915 4991 412359355 5000000 82.471871 1.0
1771948362 636 7 655360 550395 4991 417359355 5000000 83.471871 1.0
1771948363 644 8 655360 307515 4991 422359355 5000000 84.471871 1.0
1771948365 652 8 655360 64636 4991 427359356 5000001 85.4718712 1.0000002000000023
1771948365 659 7 655360 477116 4991 432359356 5000000 86.4718712 1.0
1771948367 667 8 655360 234236 4991 437359356 5000000 87.4718712 1.0
1771948367 674 7 655360 646716 4991 442359356 5000000 88.4718712 1.0
1771948369 682 8 655360 403837 4990 447359357 5000001 89.4718714 1.0000002000000023
1771948369 690 8 655360 160957 4991 452359357 5000000 90.4718714 1.0
1771948370 697 7 655360 573437 4991 457359357 5000000 91.4718714 1.0
1771948371 705 8 655360 330557 4991 462359357 5000000 92.4718714 1.0
1771948372 713 8 655360 87677 4991 467359357 5000000 93.4718714 1.0
1771948374 721 8 655360 238014 4990 472752574 5393217 94.5505148 1.0786434000000042
1771948374 728 7 655360 650494 4866 477752574 5000000 95.5505148 1.0
1771948375 736 8 655360 407614 4991 482752574 5000000 96.5505148 1.0
1771948376 744 8 655360 164734 4991 487752574 5000000 97.5505148 1.0
1771948377 751 7 655360 577215 4990 492752575 5000001 98.550515 1.0000002000000023
1771948378 759 8 655360 334335 4991 497752575 5000000 99.550515 1.0
1771948380 767 8 655360 91455 4991 502752575 5000000 100.550515 1.0
1771948380 774 7 655360 503935 4991 507752575 5000000 101.550515 1.0
1771948382 782 8 655360 261055 4991 512752575 5000000 102.550515 1.0
1771948382 790 8 655360 18176 4990 517752576 5000001 103.5505152 1.0000002000000023
1771948384 797 7 655360 430656 4991 522752576 5000000 104.5505152 1.0
1771948384 805 8 655360 187776 4991 527752576 5000000 105.5505152 1.0
1771948385 812 7 655360 600256 4991 532752576 5000000 106.5505152 1.0
1771948386 820 8 655360 357376 4991 537752576 5000000 107.5505152 1.0
1771948388 828 8 655360 507713 4991 543145793 5393217 108.6291586 1.07864339999999
1771948388 836 8 655360 264833 4991 548145793 5000000 109.6291586 1.0
1771948389 844 8 655360 21953 4991 553145793 5000000 110.6291586 1.0
1771948390 851 7 655360 434433 4991 558145793 5000000 111.6291586 1.0
1771948391 859 8 655360 191554 4990 563145794 5000001 112.6291588 1.0000002000000023
1771948392 866 7 655360 604034 4991 568145794 5000000 113.6291588 1.0
1771948393 874 8 655360 361154 4991 573145794 5000000 114.6291588 1.0
1771948395 882 8 655360 118274 4991 578145794 5000000 115.6291588 1.0
1771948395 889 7 655360 530755 4990 583145795 5000001 116.629159 1.0000002000000023
1771948397 897 8 655360 287875 4991 588145795 5000000 117.629159 1.0
1771948397 905 8 655360 44995 4991 593145795 5000000 118.629159 1.0
1771948399 912 7 655360 457475 4991 598145795 5000000 119.629159 1.0
1771948399 920 8 655360 214595 4991 603145795 5000000 120.629159 1.0
1771948401 928 8 655360 364932 4990 608539012 5393217 121.7078024 1.0786434000000042
1771948401 936 8 655360 122052 4991 613539012 5000000 122.7078024 1.0
1771948402 943 7 655360 534532 4991 618539012 5000000 123.7078024 1.0
1771948403 951 8 655360 291652 4991 623539012 5000000 124.7078024 1.0
1771948404 959 8 655360 48772 4991 628539012 5000000 125.7078024 1.0
1771948405 966 7 655360 461253 4991 633539013 5000001 126.7078026 1.0000001999999881
1771948406 974 8 655360 218373 4991 638539013 5000000 127.7078026 1.0
1771948407 981 7 655360 630853 4991 643539013 5000000 128.7078026 1.0000000000000142
1771948408 989 8 655360 387973 4991 648539013 5000000 129.7078026 1.0
1771948410 997 8 655360 145094 4990 653539014 5000001 130.7078028 1.0000001999999881
"""
# Parse lines
col0 = []
col7 = []
dt   = []

for line in raw_data.strip().split("\n"):
    parts = line.split()
    col0.append(float(parts[0]))
    col7.append(float(parts[7]))
    dt.append(float(parts[9]))

# Plot
import matplotlib.pyplot as plt

plt.figure()
plt.plot(col0, col7, '.')
plt.xlabel("Column 0")
plt.ylabel("Column 7")
plt.title("Column 7 vs Column 0")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
# Istogramma
dt=np.array(dt)
dt=dt[dt<1.07]

plt.figure()
plt.hist(dt, bins=100)
plt.xlabel("Dt")
plt.ylabel("Frequency")
plt.title(f"sampled seconds uncertainty")
plt.tight_layout()
plt.show()


In [ ]:
np.mean(dt)

In [ ]:
np.std(dt)

In [ ]:
5393216-5000000

In [ ]:
393216*2